# Bandingkan S1 dan S2 sebelum E15

S1 adalah RoBERTa dan S2 adalah BGE-M3. Notebook ini membaca dua ZIP ringkas
hasil E14 dan membandingkan prediksi validasi dari lima fold yang sama.
GPU tidak diperlukan.

Jalankan semua cell dan unggah dua file berikut saat diminta.

- `Hasil_AES_Reset_E14_S1_Ringkas_04_05.zip`
- `Hasil_AES_Reset_E14_S2_Ringkas_04_05.zip`

Jika belum punya hasil kedua skenario, jalankan
[E14 di Colab](https://colab.research.google.com/github/MalikHariyanto/Automated_Essay_Scoring/blob/codex/aes-learning/eksperimen/03_tiga_input_multi_trait/notebooks/AES_Reset_E14_S1_S2_S3_MultiTrait.ipynb).


In [ ]:
from io import BytesIO
import json
from pathlib import PurePosixPath
from zipfile import ZipFile

from IPython.display import display
import numpy as np
import pandas as pd

TRAITS = ('isi', 'organisasi', 'kebahasaan', 'mekanik')
METRICS = ('RMSE', 'Pearson', 'QWK', 'Gwet AC2')
KEYS = ['id_jawaban', 'fold', 'kelompok', 'no_soal', 'baris_excel',
        *(f'{trait}_label' for trait in TRAITS)]


def load_oof(zip_bytes):
    with ZipFile(BytesIO(zip_bytes)) as archive:
        names = [name for name in archive.namelist()
                 if PurePosixPath(name).name == '04_Prediksi_dan_Evaluasi.xlsx']
        if len(names) != 1:
            raise ValueError('Setiap ZIP ringkas harus berisi satu workbook 04.')
        contents = archive.read(names[0])

    with pd.ExcelFile(BytesIO(contents), engine='openpyxl') as excel:
        info = excel.parse('Info')
        metadata = dict(zip(info['field'], info['value']))
        config = json.loads(metadata['final_config'])
        predictions = excel.parse('Prediksi OOF').sort_values('id_jawaban').reset_index(drop=True)
        metrics = excel.parse('CV OOF per Trait').set_index('trait').loc[list(TRAITS)]
    scenario = config.get('scenario')
    if config.get('experiment') != 'Reset_E14' or scenario not in ('S1', 'S2'):
        raise ValueError('Gunakan ZIP ringkas skenario S1 atau S2 dari E14.')
    if (predictions['id_jawaban'].isna().any()
        or predictions['id_jawaban'].duplicated().any()
        or len(predictions) != int(metadata['development_rows'])
        or set(predictions['fold']) != {1, 2, 3, 4, 5}
        or predictions[KEYS].isna().any().any()):
        raise ValueError(f'Prediksi validasi {scenario} tidak lengkap.')
    return scenario, metadata, predictions, metrics


def compare_oof(archives):
    runs = {}
    for contents in archives.values():
        scenario, metadata, predictions, metrics = load_oof(contents)
        if scenario in runs:
            raise ValueError(f'Ada dua file untuk {scenario}. Unggah satu file per skenario.')
        runs[scenario] = metadata, predictions, metrics
    if set(runs) != {'S1', 'S2'}:
        raise ValueError('Diperlukan satu ZIP S1 dan satu ZIP S2.')

    info1, s1, m1 = runs['S1']
    info2, s2, m2 = runs['S2']
    for key in ('source_sha256', 'split_sha256', 'notebook_code_sha256'):
        if not info1.get(key) or info1[key] != info2.get(key):
            raise ValueError(f'Dua hasil memakai data, split, atau versi notebook berbeda ({key}).')
    if not s1[KEYS].equals(s2[KEYS]):
        raise ValueError('Jawaban, label, atau fold kedua skenario berbeda.')

    comparison = pd.concat({'S1 RoBERTa': m1[list(METRICS)], 'S2 BGE-M3': m2[list(METRICS)]}, axis=1)
    fold_rows = []
    low_score_rows = []
    for trait in TRAITS:
        label = s1[f'{trait}_label']
        residuals = [frame[f'{trait}_prediksi_mentah'] - label for frame in (s1, s2)]
        for residual, metric in zip(residuals, (m1, m2)):
            if not np.isfinite(residual).all() or not np.isclose(
                np.sqrt(residual.pow(2).mean()), metric.loc[trait, 'RMSE'], atol=1e-10, rtol=0
            ):
                raise ValueError(f'Prediksi dan RMSE pada rekap berbeda untuk {trait}.')
        for fold in sorted(s1['fold'].unique()):
            selected = s1['fold'].eq(fold)
            first, second = [np.sqrt(diff[selected].pow(2).mean()) for diff in residuals]
            fold_rows.append({'trait': trait, 'fold': fold, 'RMSE S1': first, 'RMSE S2': second,
                              'selisih S2 − S1': second - first})
        for scenario, frame in (('S1', s1), ('S2', s2)):
            category = frame[f'{trait}_prediksi_kategori']
            low_score_rows.append({'trait': trait, 'skenario': scenario,
                                   'jumlah label 1': int(label.eq(1).sum()),
                                   'label 1 terdeteksi': int((label.eq(1) & category.eq(1)).sum()),
                                   'label 2 keliru jadi 1': int((label.eq(2) & category.eq(1)).sum())})
    return comparison, pd.DataFrame(fold_rows), pd.DataFrame(low_score_rows)


## Unggah ZIP dan lihat hasil

RMSE yang lebih kecil berarti prediksi lebih dekat ke skor guru. Untuk Pearson,
QWK, dan Gwet AC2, baca nilai yang lebih tinggi bersama hasil RMSE tiap trait.
Di tabel fold, selisih positif berarti S1 punya RMSE lebih rendah.


In [ ]:
from google.colab import files

uploaded = files.upload()
comparison, folds, low_scores = compare_oof(uploaded)

print('Empat metrik per aspek · seluruh prediksi validasi')
display(comparison.round(4))
print('RMSE per fold · cek apakah arah perbedaan konsisten')
display(folds.round(4))
print('Skor 1 · cek juga berapa skor 2 yang keliru turun ke 1')
display(low_scores)


## Baca sebelum memilih perubahan S3

Cari apakah S1 dan S2 unggul pada aspek yang berbeda, lalu cek apakah pola itu
muncul di beberapa fold. Kalau ada pola yang konsisten, kita punya dasar dari
data untuk mempertimbangkan porsi encoder yang berbeda per aspek. Perbandingan
ini memakai validasi saja. Pengujian perubahan S3 dilakukan pada eksperimen
terpisah dengan S3 E13 sebagai kontrol.
